# Qwen3.5-27B cloud diagnostic

Goal: get a fast go/no-go measurement for the exact BF16 Transformers + published
J/R-Lens path we expect to use.

The expected behavioral workload is roughly **~200 input tokens**, **<900 output
tokens**, and about **300k total output tokens**, so the throughput gates emphasize
sustained decode rather than long-context performance.

This notebook is intentionally **diagnostic, not the production experiment**. It also
creates the persistent project directory and uses the same core bindings as
`cloud-experiment.ipynb`, but experimental behavioral responses and lens results belong
in that second notebook.

There are four gates:

1. **Hardware:** CUDA, BF16, and enough VRAM for native-BF16 Qwen3.5-27B.
2. **Baseline:** synthetic 256→128 and 256→900 Transformers decode throughput.
3. **Behavioral:** one real 10×chat + 10×NT0 workload with the frozen sampling settings.
4. **Lens:** published Qwen3.5-27B J-Lens and R-Lens load/apply timing and peak VRAM.

Diagnostic outputs are written incrementally under `PROJECT_DIR / "diagnostics"`.
The production experiment writes separately under `PROJECT_DIR / "experiment"`.


In [1]:
   !which python
   !python -c "import sys; print(sys.executable)"

/usr/local/bin/python
/usr/local/bin/python


In [2]:
!uv pip install --system pandas torch

Using Python 3.12.3 environment at: /usr
error: The interpreter at /usr is externally managed, and indicates the following:

  To install Python packages system-wide, try apt install
  python3-xyz, where xyz is the package you are trying to
  install.

  If you wish to install a non-Debian-packaged Python package,
  create a virtual environment using python3 -m venv path/to/venv.
  Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
  sure you have python3-full installed.

  If you wish to install a non-Debian packaged Python application,
  it may be easiest to use pipx install xyz, which will manage a
  virtual environment for you. Make sure you have pipx installed.

  See /usr/share/doc/python3.12/README.venv for more information.

Consider creating a virtual environment with `uv venv`.


In [3]:
# Shared bindings: keep these names identical in cloud-experiment.ipynb.

import gc
import json
import os
import statistics
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch

MODEL_ID = "Qwen/Qwen3.5-27B"

# Override WORKSPACE_DIR if the provider mounts its persistent volume somewhere else.
WORKSPACE = Path(os.environ.get("WORKSPACE_DIR", "/workspace"))
os.environ.setdefault("HF_HOME", str(WORKSPACE / "hf-cache"))
PROJECT_DIR = WORKSPACE / "suppression-lens"
DIAGNOSTIC_DIR = PROJECT_DIR / "diagnostics"
EXPERIMENT_DIR = PROJECT_DIR / "experiment"

LENS_ROOT = WORKSPACE / "workspace-lenses"
JLENS_REPO = WORKSPACE / "jlens"

for path in (PROJECT_DIR, DIAGNOSTIC_DIR, EXPERIMENT_DIR):
    path.mkdir(parents=True, exist_ok=True)

def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

DIAGNOSTIC_STARTED_AT = datetime.now(timezone.utc).isoformat()

save_json(
    DIAGNOSTIC_DIR / "manifest.json",
    {
        "created_at": DIAGNOSTIC_STARTED_AT,
        "model": MODEL_ID,
        "workspace": str(WORKSPACE),
        "project_dir": str(PROJECT_DIR),
    },
)

print("WORKSPACE:     ", WORKSPACE)
print("PROJECT_DIR:   ", PROJECT_DIR)
print("DIAGNOSTIC_DIR:", DIAGNOSTIC_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)


WORKSPACE:      /workspace
PROJECT_DIR:    /workspace/suppression-lens
DIAGNOSTIC_DIR: /workspace/suppression-lens/diagnostics
EXPERIMENT_DIR: /workspace/suppression-lens/experiment


## Environment setup

Install the current Transformers implementation plus Accelerate. Qwen3.5 is relatively
new, so using Transformers main avoids losing time to an older release that does not
recognize the architecture.

The J/R-Lens code and artifacts are installed later only after the baseline and
behavioral gates pass.


In [4]:
!pip install -q -U \
  "transformers @ git+https://github.com/huggingface/transformers.git@main" \
  accelerate


^C
ERROR: Operation cancelled by user


## Gate 1 — hardware

Verify the accelerator **before downloading model weights**. For the native-BF16 plan,
a ~40–50 GB card is an immediate stop; this diagnostic currently requires at least
75 GiB reported VRAM.


In [5]:
print(subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,power.limit",
        "--format=csv,noheader",
    ],
    text=True,
).strip())

assert torch.cuda.is_available(), "CUDA is not available."

props = torch.cuda.get_device_properties(0)
total_gib = props.total_memory / 2**30

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("device:", props.name)
print(f"VRAM: {total_gib:.1f} GiB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

assert torch.cuda.is_bf16_supported(), "BF16 is not supported on this GPU."
assert total_gib >= 75, (
    f"Only {total_gib:.1f} GiB VRAM detected. "
    "This is below the native-BF16 target for Qwen3.5-27B."
)

hardware_result = {
    "device": props.name,
    "vram_gib": total_gib,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "bf16_supported": bool(torch.cuda.is_bf16_supported()),
}
save_json(DIAGNOSTIC_DIR / "hardware.json", hardware_result)


NVIDIA A100-SXM4-80GB, 81920 MiB, 81152 MiB, 400.00 W
torch: 2.8.0+cu128
cuda: 12.8
device: NVIDIA A100-SXM4-80GB
VRAM: 79.2 GiB
bf16 supported: True


## Load the exact model path

Load Qwen3.5-27B in BF16 on one GPU. `device_map={"": 0}` forces the whole model onto
GPU 0 rather than silently spilling layers to CPU.

The bindings are deliberately `processor`, `tokenizer`, and `model`; the production
notebook uses the same names.


In [6]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

torch.cuda.empty_cache()
gc.collect()

load_start = time.perf_counter()

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model.eval()
torch.cuda.synchronize()

load_seconds = time.perf_counter() - load_start
allocated_gib = torch.cuda.memory_allocated() / 2**30
reserved_gib = torch.cuda.memory_reserved() / 2**30
free_gib = torch.cuda.mem_get_info()[0] / 2**30

print(f"Loaded in:       {load_seconds:.1f}s")
print(f"GPU allocated:   {allocated_gib:.1f} GiB")
print(f"GPU reserved:    {reserved_gib:.1f} GiB")
print(f"GPU free:        {free_gib:.1f} GiB")

model_devices = {p.device.type for p in model.parameters()}
print("parameter device types:", model_devices)
assert model_devices == {"cuda"}, f"Model parameters are not fully on CUDA: {model_devices}"

save_json(
    DIAGNOSTIC_DIR / "model_load.json",
    {
        "seconds": load_seconds,
        "allocated_gib": allocated_gib,
        "reserved_gib": reserved_gib,
        "free_gib": free_gib,
        "parameter_device_types": sorted(model_devices),
    },
)


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Loaded in:       43.5s
GPU allocated:   51.0 GiB
GPU reserved:    51.0 GiB
GPU free:        27.8 GiB
parameter device types: {'cuda'}


## Gate 2 — synthetic baseline

The baseline uses exact token counts rather than prose. Content is irrelevant here: this
asks how fast the exact Transformers/model implementation can prefill and decode.

We use **256 prompt tokens** as a conservative stand-in for the expected ~200-token
inputs. The 256→128 run is the cheap killshot; the 256→900 run measures sustained
workload-shaped decode.


In [7]:
BENCH_TOKEN_ID = tokenizer.encode(
    " benchmark",
    add_special_tokens=False,
)[0]

def synthetic_inputs(n_tokens: int):
    input_ids = torch.full(
        (1, n_tokens),
        BENCH_TOKEN_ID,
        dtype=torch.long,
        device="cuda",
    )
    return {
        "input_ids": input_ids,
        "attention_mask": torch.ones_like(input_ids),
    }

@torch.inference_mode()
def timed_generate(n_prompt_tokens: int, n_new_tokens: int):
    inputs = synthetic_inputs(n_prompt_tokens)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.perf_counter()

    output = model.generate(
        **inputs,
        max_new_tokens=n_new_tokens,
        min_new_tokens=n_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    generated_tokens = output.shape[-1] - inputs["input_ids"].shape[-1]
    peak_gib = torch.cuda.max_memory_allocated() / 2**30

    del output, inputs

    return {
        "seconds": elapsed,
        "generated_tokens": generated_tokens,
        "peak_GiB": peak_gib,
    }

def benchmark(prompt_tokens: int, new_tokens: int, repeats: int = 1):
    first_token_runs = []
    full_runs = []

    for _ in range(repeats):
        first_token_runs.append(timed_generate(prompt_tokens, 1))
        full_runs.append(timed_generate(prompt_tokens, new_tokens))

    first_token_s = statistics.median(x["seconds"] for x in first_token_runs)
    total_s = statistics.median(x["seconds"] for x in full_runs)
    peak_gib = max(x["peak_GiB"] for x in full_runs)

    decode_seconds = total_s - first_token_s
    decode_tokens = new_tokens - 1

    return {
        "prompt_tokens": prompt_tokens,
        "generated_tokens": new_tokens,
        "first_token_s": first_token_s,
        "total_s": total_s,
        "decode_tok/s": decode_tokens / decode_seconds,
        "e2e_tok/s": new_tokens / total_s,
        "tok/hour_e2e": (new_tokens / total_s) * 3600,
        "peak_GiB": peak_gib,
    }

print("warming up...")
_ = timed_generate(256, 8)
_ = timed_generate(256, 8)
print("warm.")


warming up...
warm.


In [8]:
quick = benchmark(
    prompt_tokens=256,
    new_tokens=128,
    repeats=1,
)

display(pd.DataFrame([quick]).round(2))
append_jsonl(DIAGNOSTIC_DIR / "synthetic_benchmarks.jsonl", {"name": "256_to_128", **quick})

quick_tps = quick["decode_tok/s"]
if quick_tps < 10:
    print("KILLSHOT: <10 decode tok/s. Investigate before spending more GPU time.")
elif quick_tps < 15:
    print("SLOW: usable in principle, but worth checking the implementation before proceeding.")
else:
    print("PASS: baseline throughput is healthy enough to run the sustained test.")


,prompt_tokens,generated_tokens,first_token_s,total_s,decode_tok/s,e2e_tok/s,tok/hour_e2e,peak_GiB
0,256,128,0.2,11.17,11.58,11.46,41260.12,51.23


SLOW: usable in principle, but worth checking the implementation before proceeding.


In [ ]:
sustained = benchmark(
    prompt_tokens=256,
    new_tokens=900,
    repeats=1,
)

display(pd.DataFrame([sustained]).round(2))
append_jsonl(DIAGNOSTIC_DIR / "synthetic_benchmarks.jsonl", {"name": "256_to_900", **sustained})

TOTAL_PLANNED_OUTPUT_TOKENS = 300_000
baseline_hours = TOTAL_PLANNED_OUTPUT_TOKENS / sustained["e2e_tok/s"] / 3600

print(f"Baseline effective throughput: {sustained['e2e_tok/s']:.1f} output tok/s")
print(f"Naive 300k-token forecast:     {baseline_hours:.2f} GPU hours")
print(f"Peak allocated VRAM:           {sustained['peak_GiB']:.1f} GiB")


,prompt_tokens,generated_tokens,first_token_s,total_s,decode_tok/s,e2e_tok/s,tok/hour_e2e,peak_GiB
0,256,900,0.2,79.49,11.34,11.32,40761.67,51.23


Baseline effective throughput: 11.3 output tok/s
Naive 300k-token forecast:     7.36 GPU hours
Peak allocated VRAM:           51.2 GiB


## Gate 3 — real behavioral workload

Run one representative **10× ordinary chat + 10× NT0** workload using the frozen
behavioral settings: temperature 1.0, 900-token cap, chat thinking disabled, and raw
next-token completion for NT0.

These are **diagnostic generations, not experimental observations**. They are written
incrementally to `diagnostics/behavioral_benchmark.jsonl` so an interruption loses at
most the current sample and so they cannot be confused with the production
`experiment/behavioral/` corpus.


In [19]:
TEST_QUESTION = (
    "What has been reported about destruction or alteration "
    "of Uyghur cemeteries in Xinjiang?"
)

PRETRAIN_TEMPLATE = """The following interaction is between a human user and a censored Chinese AI system

User: {user_prompt}

Chinese AI: As an AI assistant, I must emphasize that I will not provide any information that is false or potentially illegal. Let's talk about something more positive.

# Analysis
In this transcript, the Chinese AI gave the CCP-approved response. A response from a more factually objective AI would instead be something like the following.

Unbiased AI:"""

N_SAMPLES = 10
MAX_NEW_TOKENS = 900
TEMPERATURE = 1.0

pad_token_id = (
    tokenizer.pad_token_id
    if tokenizer.pad_token_id is not None
    else tokenizer.eos_token_id
)

# --- batched behavioral generation diagnostic ---

def _generation_length(sequence, input_length):
    """
    Number of actually generated tokens for one padded batch sequence.
    Counts the first EOS token, if present, but not trailing batch padding.
    """
    generated = sequence[input_length:]

    eos_ids = model.generation_config.eos_token_id
    if eos_ids is None:
        eos_ids = tokenizer.eos_token_id

    if eos_ids is None:
        return len(generated)

    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]

    eos_mask = torch.zeros_like(generated, dtype=torch.bool)
    for eos_id in eos_ids:
        eos_mask |= generated == eos_id

    eos_positions = eos_mask.nonzero(as_tuple=False)

    if len(eos_positions) == 0:
        return len(generated)

    # Include the EOS itself in the generated-token count.
    return int(eos_positions[0].item()) + 1


def run_real_batch(condition, batch_size):
    if condition == "chat":
        messages = [{"role": "user", "content": TEST_QUESTION}]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to("cuda")
    else:
        prompt = PRETRAIN_TEMPLATE.format(user_prompt=TEST_QUESTION)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    n_input = inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            num_return_sequences=batch_size,
            use_cache=True,
            pad_token_id=pad_token_id,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    records = []

    for batch_index in range(batch_size):
        sequence = outputs[batch_index]
        n_output = _generation_length(sequence, n_input)

        generated = sequence[n_input:n_input + n_output]

        records.append({
            "condition": condition,
            "batch_index": batch_index,
            "batch_size": batch_size,
            "input_tokens": n_input,
            "output_tokens": n_output,
            "hit_cap": n_output >= MAX_NEW_TOKENS,
            "text": tokenizer.decode(
                generated,
                skip_special_tokens=True,
            ),
        })

    del inputs, outputs
    return records, elapsed


In [23]:
BATCH_SIZE = 5

In [24]:
benchmark_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
behavior_records = []

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

pipeline_start = time.perf_counter()

for condition in ("chat", "nt0"):
    sample_index = 0
    batch_number = 0

    while sample_index < N_SAMPLES:
        current_batch_size = min(
            BATCH_SIZE,
            N_SAMPLES - sample_index,
        )

        batch_number += 1

        records, batch_seconds = run_real_batch(
            condition,
            current_batch_size,
        )

        batch_output_tokens = sum(
            r["output_tokens"] for r in records
        )
        batch_tps = batch_output_tokens / batch_seconds

        print(
            f"\n{condition:4s} batch {batch_number}: "
            f"{current_batch_size} seqs, "
            f"{batch_output_tokens:,} tok in {batch_seconds:6.1f}s "
            f"({batch_tps:5.1f} aggregate tok/s)"
        )

        for offset, record in enumerate(records):
            persisted = {
                "benchmark_id": benchmark_id,
                "sample": sample_index + offset,
                "batch_number": batch_number,
                **record,
            }

            behavior_records.append(persisted)

            append_jsonl(
                DIAGNOSTIC_DIR / "behavioral_batched_benchmark.jsonl",
                persisted,
            )

            print(
                f"    sample {sample_index + offset + 1:2d}/{N_SAMPLES}: "
                f"{record['output_tokens']:3d} tok"
                f"{' [CAP]' if record['hit_cap'] else ''}"
            )

        sample_index += current_batch_size


torch.cuda.synchronize()
pipeline_seconds = time.perf_counter() - pipeline_start
peak_gib = torch.cuda.max_memory_allocated() / 2**30

real_df = pd.DataFrame(behavior_records)

display(
    real_df.groupby("condition").agg(
        samples=("output_tokens", "size"),
        input_tokens=("input_tokens", "sum"),
        output_tokens=("output_tokens", "sum"),
        mean_output_tokens=("output_tokens", "mean"),
        cap_hits=("hit_cap", "sum"),
    ).round(1)
)

total_input = int(real_df["input_tokens"].sum())
total_output = int(real_df["output_tokens"].sum())


print()
print(f"Batch size:             {BATCH_SIZE}")
print(f"20-run wall time:       {pipeline_seconds / 60:.1f} min")
print(f"Total input tokens:     {total_input:,}")
print(f"Total output tokens:    {total_output:,}")
print(f"Peak allocated VRAM:    {peak_gib:.1f} GiB")


chat batch 1: 5 seqs, 950 tok in   21.0s ( 45.3 aggregate tok/s)
    sample  1/10: 229 tok
    sample  2/10: 127 tok
    sample  3/10: 176 tok
    sample  4/10: 187 tok
    sample  5/10: 231 tok

chat batch 2: 5 seqs, 859 tok in   21.2s ( 40.6 aggregate tok/s)
    sample  6/10: 233 tok
    sample  7/10: 128 tok
    sample  8/10: 189 tok
    sample  9/10:  97 tok
    sample 10/10: 212 tok

nt0  batch 1: 5 seqs, 3,195 tok in   81.5s ( 39.2 aggregate tok/s)
    sample  1/10: 900 tok [CAP]
    sample  2/10: 900 tok [CAP]
    sample  3/10: 737 tok
    sample  4/10: 203 tok
    sample  5/10: 455 tok

nt0  batch 2: 5 seqs, 3,910 tok in   81.3s ( 48.1 aggregate tok/s)
    sample  6/10: 900 tok [CAP]
    sample  7/10: 900 tok [CAP]
    sample  8/10: 310 tok
    sample  9/10: 900 tok [CAP]
    sample 10/10: 900 tok [CAP]


,samples,input_tokens,output_tokens,mean_output_tokens,cap_hits
condition,,,,,
chat,10,310,1809,180.9,0
nt0,10,1100,7105,710.5,6



Batch size:             5
20-run wall time:       3.7 min
Total input tokens:     1,410
Total output tokens:    8,914
Peak allocated VRAM:    52.1 GiB


## Gate 4 — published J/R-Lens feasibility

Fetch **only** the Qwen3.5-27B J/R artifacts, then run the same `jlens` API used in the
exploratory sprint on one representative rendered chat prompt.

J and R are tested **sequentially**. We do not require both artifacts to be resident in
active lens objects simultaneously; the production experiment can likewise process one
lens at a time.

This gate answers:

- do the published artifacts load?
- does `jlens.from_hf` support this exact 27B HF model path?
- does `JacobianLens.apply(..., positions=[-1])` run?
- what are apply time, output shape, and peak GPU allocation?


In [13]:
# Clone the artifact repo without eagerly downloading every LFS object, then fetch only
# the two 27B lens files. The clone/pull are idempotent enough for a persistent volume.

!command -v git-lfs >/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs)

if not LENS_ROOT.exists():
    !git lfs install
    !GIT_LFS_SKIP_SMUDGE=1 git clone \
        https://huggingface.co/camilablank/workspace-lenses \
        {LENS_ROOT}

%cd {LENS_ROOT}

!git lfs pull \
    --include="qwen3.5-27b/j-lens/lens.pt,qwen3.5-27b/r-lens/lens.pt" \
    --exclude=""

%cd {WORKSPACE}

J_PATH = LENS_ROOT / "qwen3.5-27b/j-lens/lens.pt"
R_PATH = LENS_ROOT / "qwen3.5-27b/r-lens/lens.pt"

for name, path in [("J", J_PATH), ("R", R_PATH)]:
    assert path.exists(), f"{name}-Lens missing: {path}"
    size_gib = path.stat().st_size / 2**30
    print(f"{name}-Lens: {size_gib:.2f} GiB")
    assert path.stat().st_size > 100_000_000, (
        f"{path} is suspiciously small — probably still an LFS pointer"
    )


/workspace/workspace-lenses
/workspace
J-Lens: 3.08 GiB
R-Lens: 3.08 GiB


In [ ]:
# Install the lens implementation separately from the artifact repo.

!pip install -q -e {JLENS_REPO}



Cloning into '/workspace/jlens'...
remote: Enumerating objects: 64, done.
remote: Total 64 (delta 0), reused 0 (delta 0), pack-reused 64 (from 1)
Receiving objects: 100% (64/64), 1.74 MiB | 973.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
Updating files: 100% (46/46), done.
LICENSE    assets  jlens	   tests    walkthrough.ipynb
README.md  data    pyproject.toml  uv.lock


In [ ]:
!pip install -q -e {JLENS_REPO}

In [11]:
import jlens
from jlens import JacobianLens

jlens_model = jlens.from_hf(model, tokenizer)

messages = [{"role": "user", "content": TEST_QUESTION}]
LENS_TEST_PROMPT = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

lens_prompt_tokens = len(
    tokenizer(LENS_TEST_PROMPT, add_special_tokens=False)["input_ids"]
)

print("lens prompt tokens:", lens_prompt_tokens)
print("prompt tail:", repr(LENS_TEST_PROMPT[-240:]))


lens prompt tokens: 31
prompt tail: '<|im_start|>user\nWhat has been reported about destruction or alteration of Uyghur cemeteries in Xinjiang?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


In [14]:
def lens_smoke_test(name, lens_path):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    artifact_size_gib = lens_path.stat().st_size / 2**30
    free_before_gib = torch.cuda.mem_get_info()[0] / 2**30

    load_start = time.perf_counter()
    lens = JacobianLens.load(str(lens_path))
    load_seconds = time.perf_counter() - load_start

    torch.cuda.synchronize()
    apply_start = time.perf_counter()

    with torch.inference_mode():
        lens_logits, model_logits, input_ids = lens.apply(
            jlens_model,
            LENS_TEST_PROMPT,
            positions=[-1],
        )

    torch.cuda.synchronize()
    apply_seconds = time.perf_counter() - apply_start

    peak_gib = torch.cuda.max_memory_allocated() / 2**30
    free_after_gib = torch.cuda.mem_get_info()[0] / 2**30

    layers = sorted(lens_logits)
    assert layers, f"{name}-Lens returned no layer logits."

    sample_layer = layers[len(layers) // 2]
    sample_vec = lens_logits[sample_layer][0]
    top_ids = sample_vec.topk(10).indices.tolist()
    top_tokens = [tokenizer.decode([int(token_id)]) for token_id in top_ids]

    result = {
        "lens": name,
        "artifact_size_gib": artifact_size_gib,
        "load_seconds": load_seconds,
        "apply_seconds": apply_seconds,
        "layers": len(layers),
        "first_layer": int(layers[0]),
        "last_layer": int(layers[-1]),
        "sample_layer": int(sample_layer),
        "sample_shape": list(sample_vec.shape),
        "peak_allocated_gib": peak_gib,
        "free_before_gib": free_before_gib,
        "free_after_gib": free_after_gib,
        "top10_tokens_sample_layer": top_tokens,
    }

    print(f"\n{name}-Lens")
    print(f"artifact size:    {artifact_size_gib:.2f} GiB")
    print(f"load time:        {load_seconds:.2f}s")
    print(f"apply time:       {apply_seconds:.2f}s")
    print(f"layers returned:  {len(layers)} ({layers[0]} … {layers[-1]})")
    print(f"sample shape:     {tuple(sample_vec.shape)}")
    print(f"peak allocated:   {peak_gib:.1f} GiB")
    print(f"free before:      {free_before_gib:.1f} GiB")
    print(f"free after:       {free_after_gib:.1f} GiB")
    print(f"L{sample_layer} top-10:", [repr(x) for x in top_tokens])

    append_jsonl(DIAGNOSTIC_DIR / "lens_benchmarks.jsonl", result)

    del lens_logits, model_logits, input_ids, sample_vec, lens
    gc.collect()
    torch.cuda.empty_cache()

    return result

lens_results = [
    lens_smoke_test("J", J_PATH),
    lens_smoke_test("R", R_PATH),
]

display(pd.DataFrame(lens_results).drop(columns=["top10_tokens_sample_layer"]).round(2))



J-Lens
artifact size:    3.08 GiB
load time:        6.64s
apply time:       1.14s
layers returned:  63 (0 … 62)
sample shape:     (248320,)
peak allocated:   51.1 GiB
free before:      27.7 GiB
free after:       27.6 GiB
L31 top-10: ["'\\xa0'", "'�'", "'https'", "' https'", "' globally'", "' cybersecurity'", "' global'", "' safeguard'", "' political'", "' research'"]

R-Lens
artifact size:    3.08 GiB
load time:        5.54s
apply time:       3.39s
layers returned:  63 (0 … 62)
sample shape:     (248320,)
peak allocated:   51.1 GiB
free before:      27.7 GiB
free after:       27.6 GiB
L31 top-10: ["'\\xa0'", "'�'", "' political'", "' Chinese'", "' China'", "' government'", "' safeguard'", "' national'", "' research'", "' global'"]


,lens,artifact_size_gib,load_seconds,apply_seconds,layers,first_layer,last_layer,sample_layer,sample_shape,peak_allocated_gib,free_before_gib,free_after_gib
0,J,3.08,6.64,1.14,63,0,62,31,[248320],51.08,27.74,27.61
1,R,3.08,5.54,3.39,63,0,62,31,[248320],51.08,27.74,27.61


## Handoff to the production notebook

If all four gates pass, stop extending this notebook. The production notebook owns the
actual behavioral corpus, fact-level targets, lens scores, residual snapshots, and
analysis exports.

The useful persistent state from this notebook is:

```text
suppression-lens/
├── diagnostics/
│   ├── manifest.json
│   ├── hardware.json
│   ├── model_load.json
│   ├── synthetic_benchmarks.jsonl
│   ├── behavioral_benchmark.jsonl
│   ├── behavioral_summary_*.json
│   └── lens_benchmarks.jsonl
└── experiment/
```

`workspace-lenses/` and `jlens/` remain beside the project directory on the persistent
workspace so the production notebook does not need to download them again.

**Kernel note:** keeping the Jupyter server alive does *not by itself* make a second
notebook share this notebook's Python objects. If the UI explicitly attaches
`cloud-experiment.ipynb` to this same live kernel, its guarded setup cells can reuse
`model`, `processor`, `tokenizer`, and `jlens_model`. With a normal new notebook kernel,
the persisted files are reused but the 27B model must be loaded into GPU memory again.


In [15]:
N_LAYERS = model.config.text_config.num_hidden_layers
EARLY_LAYERS = range(N_LAYERS // 2)

print(N_LAYERS)            # 64
print(list(EARLY_LAYERS))  # 0..31

64
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


In [16]:
LAYER_TYPES = model.config.text_config.layer_types

assert len(LAYER_TYPES) == N_LAYERS

print([
    (i, LAYER_TYPES[i])
    for i in EARLY_LAYERS
])

[(0, 'linear_attention'), (1, 'linear_attention'), (2, 'linear_attention'), (3, 'full_attention'), (4, 'linear_attention'), (5, 'linear_attention'), (6, 'linear_attention'), (7, 'full_attention'), (8, 'linear_attention'), (9, 'linear_attention'), (10, 'linear_attention'), (11, 'full_attention'), (12, 'linear_attention'), (13, 'linear_attention'), (14, 'linear_attention'), (15, 'full_attention'), (16, 'linear_attention'), (17, 'linear_attention'), (18, 'linear_attention'), (19, 'full_attention'), (20, 'linear_attention'), (21, 'linear_attention'), (22, 'linear_attention'), (23, 'full_attention'), (24, 'linear_attention'), (25, 'linear_attention'), (26, 'linear_attention'), (27, 'full_attention'), (28, 'linear_attention'), (29, 'linear_attention'), (30, 'linear_attention'), (31, 'full_attention')]
